# 01 - Introducao ao Tool Use
> Como dar ferramentas ao Claude e processar chamadas

**Modulo:** `03_tool_use` | **Feito com:** [Claude](https://claude.ai) (Anthropic)

---


In [ ]:
import os
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
HAIKU  = 'claude-haiku-4-5-20251001'
SONNET = 'claude-sonnet-4-5'

def ask(prompt, system=None, model=HAIKU, max_tokens=1024):
    kw = dict(model=model, max_tokens=max_tokens,
              messages=[{'role':'user','content':prompt}])
    if system: kw['system'] = system
    return client.messages.create(**kw).content[0].text

print('Pronto!')

In [ ]:
tools = [{'name':'calcular',
    'description':'Calcula expressoes matematicas Python.',
    'input_schema':{'type':'object',
        'properties':{'expr':{'type':'string'}},'required':['expr']}}]

def calcular(expr):
    try: return str(eval(expr, {'__builtins__':{}}, {}))
    except Exception as e: return f'Erro: {e}'

def loop(pergunta):
    msgs=[{'role':'user','content':pergunta}]
    while True:
        r=client.messages.create(model=HAIKU,max_tokens=512,tools=tools,messages=msgs)
        if r.stop_reason=='end_turn': return r.content[0].text
        msgs.append({'role':'assistant','content':r.content})
        res=[]
        for b in r.content:
            if b.type=='tool_use':
                res.append({'type':'tool_result','tool_use_id':b.id,'content':calcular(**b.input)})
        msgs.append({'role':'user','content':res})

print(loop('Quanto e 15% de 840 mais 72 dividido por 8?'))

## Exercicios

### Exercicio 1 — Nova ferramenta: conversor de temperatura
Crie uma tool `converter_temp` que converte Celsius para Fahrenheit (e vice-versa).
Integre no loop agentico e teste com: *"Quanto e 38°C em Fahrenheit?"*

In [ ]:
# Exercicio 1 — Conversor de temperatura

tools_temp = [
    {
        'name': 'converter_temp',
        'description': 'Converte temperatura entre Celsius e Fahrenheit. Use direction="C_to_F" ou "F_to_C".',
        'input_schema': {
            'type': 'object',
            'properties': {
                'valor': {'type': 'number', 'description': 'Valor da temperatura'},
                'direction': {
                    'type': 'string',
                    'enum': ['C_to_F', 'F_to_C'],
                    'description': 'Direcao da conversao'
                }
            },
            'required': ['valor', 'direction']
        }
    }
]

def converter_temp(valor, direction):
    if direction == 'C_to_F':
        return str(valor * 9/5 + 32)
    elif direction == 'F_to_C':
        return str((valor - 32) * 5/9)
    return 'Erro: direction deve ser C_to_F ou F_to_C'

def loop_temp(pergunta):
    msgs = [{'role': 'user', 'content': pergunta}]
    while True:
        r = client.messages.create(model=HAIKU, max_tokens=512, tools=tools_temp, messages=msgs)
        if r.stop_reason == 'end_turn':
            return r.content[0].text
        msgs.append({'role': 'assistant', 'content': r.content})
        res = []
        for b in r.content:
            if b.type == 'tool_use':
                resultado = converter_temp(**b.input)
                res.append({'type': 'tool_result', 'tool_use_id': b.id, 'content': resultado})
        msgs.append({'role': 'user', 'content': res})

# Teste
print(loop_temp('Quanto e 38°C em Fahrenheit?'))
print()
print(loop_temp('Converte 100°F pra Celsius'))

### Exercicio 2 — Combinando tools: calculadora + conversor
Monte um loop que disponibiliza DUAS tools ao mesmo tempo (`calcular` e `converter_temp`).
Teste com: *"Quanto e 20% de 180 graus Celsius em Fahrenheit?"*

In [ ]:
# Exercicio 2 — Duas tools no mesmo loop

tools_combo = [
    {
        'name': 'calcular',
        'description': 'Calcula expressoes matematicas Python.',
        'input_schema': {
            'type': 'object',
            'properties': {'expr': {'type': 'string'}},
            'required': ['expr']
        }
    },
    {
        'name': 'converter_temp',
        'description': 'Converte temperatura entre Celsius e Fahrenheit. Use direction="C_to_F" ou "F_to_C".',
        'input_schema': {
            'type': 'object',
            'properties': {
                'valor': {'type': 'number', 'description': 'Valor da temperatura'},
                'direction': {
                    'type': 'string',
                    'enum': ['C_to_F', 'F_to_C'],
                    'description': 'Direcao da conversao'
                }
            },
            'required': ['valor', 'direction']
        }
    }
]

# Dispatcher: mapeia nome da tool -> funcao Python
dispatch = {
    'calcular': lambda **kw: calcular(**kw),
    'converter_temp': lambda **kw: converter_temp(**kw),
}

def loop_combo(pergunta):
    msgs = [{'role': 'user', 'content': pergunta}]
    while True:
        r = client.messages.create(model=HAIKU, max_tokens=512, tools=tools_combo, messages=msgs)
        if r.stop_reason == 'end_turn':
            return r.content[0].text
        msgs.append({'role': 'assistant', 'content': r.content})
        res = []
        for b in r.content:
            if b.type == 'tool_use':
                fn = dispatch.get(b.name)
                resultado = fn(**b.input) if fn else f'Erro: tool {b.name} nao encontrada'
                print(f'  🔧 {b.name}({b.input}) → {resultado}')
                res.append({'type': 'tool_result', 'tool_use_id': b.id, 'content': resultado})
        msgs.append({'role': 'user', 'content': res})

# Teste — requer as duas tools em sequencia
print(loop_combo('Quanto e 20% de 180 graus Celsius em Fahrenheit?'))

### Exercicio 3 — Tool com validacao: buscador de CEP
Crie uma tool `buscar_cep` que recebe um CEP (string de 8 digitos) e retorna dados ficticios de endereco.
A tool deve **validar** se o CEP tem 8 digitos numericos antes de processar.
Teste com CEPs validos e invalidos.

In [ ]:
# Exercicio 3 — Tool com validacao: buscador de CEP
import json, re

# Base simulada de CEPs
CEP_DB = {
    '01001000': {'rua': 'Praca da Se', 'bairro': 'Se', 'cidade': 'Sao Paulo', 'estado': 'SP'},
    '90010000': {'rua': 'Rua dos Andradas', 'bairro': 'Centro Historico', 'cidade': 'Porto Alegre', 'estado': 'RS'},
    '20040020': {'rua': 'Av. Rio Branco', 'bairro': 'Centro', 'cidade': 'Rio de Janeiro', 'estado': 'RJ'},
}

tools_cep = [
    {
        'name': 'buscar_cep',
        'description': 'Busca endereco a partir de um CEP brasileiro (8 digitos, sem traco).',
        'input_schema': {
            'type': 'object',
            'properties': {
                'cep': {'type': 'string', 'description': 'CEP com 8 digitos numericos, sem traco'}
            },
            'required': ['cep']
        }
    }
]

def buscar_cep(cep):
    # Validacao: apenas 8 digitos numericos
    cep_limpo = cep.replace('-', '').replace('.', '').strip()
    if not re.fullmatch(r'\d{8}', cep_limpo):
        return json.dumps({'erro': f'CEP invalido: "{cep}". Deve ter exatamente 8 digitos numericos.'})
    dados = CEP_DB.get(cep_limpo)
    if dados:
        return json.dumps(dados, ensure_ascii=False)
    return json.dumps({'erro': f'CEP {cep_limpo} nao encontrado na base.'})

def loop_cep(pergunta):
    msgs = [{'role': 'user', 'content': pergunta}]
    while True:
        r = client.messages.create(model=HAIKU, max_tokens=512, tools=tools_cep, messages=msgs)
        if r.stop_reason == 'end_turn':
            return r.content[0].text
        msgs.append({'role': 'assistant', 'content': r.content})
        res = []
        for b in r.content:
            if b.type == 'tool_use':
                resultado = buscar_cep(**b.input)
                print(f'  🔧 buscar_cep({b.input}) → {resultado}')
                res.append({'type': 'tool_result', 'tool_use_id': b.id, 'content': resultado})
        msgs.append({'role': 'user', 'content': res})

# Testes
print('--- CEP valido ---')
print(loop_cep('Qual o endereco do CEP 90010-000?'))
print()
print('--- CEP invalido ---')
print(loop_cep('Busca o CEP abc123'))
print()
print('--- CEP nao encontrado ---')
print(loop_cep('Me da o endereco do CEP 99999999'))

### Exercicio 4 — Inspecionando a resposta crua
Faca uma chamada com tools mas **sem processar** o tool_use. Inspecione o objeto de resposta
para entender a estrutura: `stop_reason`, `content` (blocos de texto vs tool_use), `id`, `name`, `input`.
Isso ajuda a debugar loops agenticos.

In [ ]:
# Exercicio 4 — Inspecionando a resposta crua (sem processar o tool_use)

response = client.messages.create(
    model=HAIKU,
    max_tokens=512,
    tools=tools,  # usa a tool 'calcular' do inicio
    messages=[{'role': 'user', 'content': 'Quanto e 2 elevado a 10?'}]
)

print(f'stop_reason: {response.stop_reason}')
print(f'Quantidade de blocos no content: {len(response.content)}')
print()

for i, bloco in enumerate(response.content):
    print(f'--- Bloco {i} ---')
    print(f'  type: {bloco.type}')
    if bloco.type == 'text':
        print(f'  text: {bloco.text}')
    elif bloco.type == 'tool_use':
        print(f'  id:    {bloco.id}')
        print(f'  name:  {bloco.name}')
        print(f'  input: {bloco.input}')
    print()

print('💡 Quando stop_reason="tool_use", o modelo QUER chamar uma tool.')
print('   Voce deve executar a funcao e devolver o resultado como tool_result.')

### Exercicio 5 — Desafio: loop generico reutilizavel
Crie uma funcao `loop_generico(pergunta, tools_list, dispatch_dict)` que funciona com **qualquer** conjunto de tools.
Teste reutilizando todas as tools que voce criou nos exercicios anteriores.

In [ ]:
# Exercicio 5 — Loop generico reutilizavel

def loop_generico(pergunta, tools_list, dispatch_dict, model=HAIKU, max_tokens=512, verbose=True):
    """Loop agentico generico que funciona com qualquer conjunto de tools."""
    msgs = [{'role': 'user', 'content': pergunta}]
    iteracao = 0

    while True:
        iteracao += 1
        r = client.messages.create(model=model, max_tokens=max_tokens, tools=tools_list, messages=msgs)

        if r.stop_reason == 'end_turn':
            if verbose:
                print(f'  ✅ Resposta final (apos {iteracao} iteracao(es))')
            return r.content[0].text

        msgs.append({'role': 'assistant', 'content': r.content})
        res = []
        for b in r.content:
            if b.type == 'tool_use':
                fn = dispatch_dict.get(b.name)
                if fn:
                    resultado = fn(**b.input)
                else:
                    resultado = f'Erro: tool "{b.name}" nao registrada no dispatch.'
                if verbose:
                    print(f'  🔧 [{iteracao}] {b.name}({b.input}) → {resultado}')
                res.append({'type': 'tool_result', 'tool_use_id': b.id, 'content': resultado})
        msgs.append({'role': 'user', 'content': res})

# Registra TODAS as tools e funcoes dos exercicios anteriores
todas_tools = tools_combo + tools_cep  # calcular + converter_temp + buscar_cep

todas_funcoes = {
    'calcular': lambda **kw: calcular(**kw),
    'converter_temp': lambda **kw: converter_temp(**kw),
    'buscar_cep': lambda **kw: buscar_cep(**kw),
}

# Teste 1: usa calcular
print('=== Teste com calcular ===')
print(loop_generico('Quanto e 15 * 37?', todas_tools, todas_funcoes))
print()

# Teste 2: usa converter_temp
print('=== Teste com converter_temp ===')
print(loop_generico('Converte 72°F pra Celsius', todas_tools, todas_funcoes))
print()

# Teste 3: usa buscar_cep
print('=== Teste com buscar_cep ===')
print(loop_generico('Qual endereco do CEP 01001-000?', todas_tools, todas_funcoes))
print()

# Teste 4: pode precisar de multiplas tools
print('=== Teste combinado ===')
print(loop_generico('Calcule 25% de 200, converta o resultado para Fahrenheit e busque o CEP 20040020', todas_tools, todas_funcoes))

## Proximos passos
- Proximo notebook do modulo
- [docs.anthropic.com](https://docs.anthropic.com)
